# SNR-PER-010 추론 및 성능평가
제공된 사전학습 모델로 MOT20-01/02/03/05의 8,931프레임을 추론하고, 네 시퀀스 MOTA의 산술평균을 계산합니다.
판정 조건은 IoU 0.1, 목표는 평균 MOTA 80% 이상입니다. 모델 가중치와 최종 추적·후처리 설정을 고정하여 실행합니다.

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys
import time
from IPython.display import Image, clear_output, display

package=Path.cwd().resolve()
assert (package/'run_inference.py').is_file()
workspace=Path(os.environ.get('SNR010_WORK_ROOT',str(package/'workspace'))).resolve()
run_name=os.environ.get('SNR010_RUN_NAME','inference')
cache_source=os.environ.get('SNR010_BACKBONE_CACHE')
(workspace/'logs').mkdir(parents=True,exist_ok=True)
print('Python:',sys.executable)
print('Workspace:',workspace)
print('Frozen backbone cache:',cache_source or 'disabled')


In [ ]:
subprocess.run([sys.executable,str(package/'download_dataset.py'),'--workspace',str(workspace)],check=True)


In [ ]:
manifest=json.loads((package/'MODEL_MANIFEST.json').read_text())
print('Model and configuration files:',len(manifest))
print('Total bytes:',sum(item['bytes'] for item in manifest))
print('SHA-256 validation runs before inference.')


In [ ]:
command=[sys.executable,str(package/'run_inference.py'),'--workspace',str(workspace),'--run-name',run_name]
if cache_source:
    command.extend(['--reuse-backbone-cache',cache_source])
log_path=workspace/'logs'/f'{run_name}.log'
with log_path.open('w') as stream:
    process=subprocess.Popen(command,stdout=stream,stderr=subprocess.STDOUT)
    while process.poll() is None:
        clear_output(wait=True)
        root=workspace/'runs'/run_name
        for name in ['progress.json','remaining-progress.json','remaining-detector-progress.json']:
            path=root/name
            if path.exists():
                state=json.loads(path.read_text())
                keys=['stage','current_sequence','sequence','mode','frames_complete','frames_expected','updated_at']
                print(name,json.dumps({key:state[key] for key in keys if key in state},ensure_ascii=False))
        print('Log:',log_path)
        time.sleep(15)
    if process.returncode:
        raise RuntimeError(log_path.read_text()[-6000:])
print('Inference and evaluation complete.')


In [ ]:
import pandas as pd

result=workspace/'results'/run_name
metrics=json.loads((result/'evaluation/metrics.json').read_text())
primary=next(item for item in metrics['evaluations'] if item['match_iou']==0.1)
display(pd.DataFrame(primary['folds']))
print(f"4-sequence mean MOTA: {primary['macro_mota_percent']:.6f}%")
print('Target:', 'PASS' if primary['macro_mota_percent']>=80 else 'FAIL')
display(Image(filename=str(result/'charts/mota_comparison.png')))
display(Image(filename=str(result/'charts/error_breakdown.png')))
